In [1]:
   print("hello")

hello


In [3]:
import pandas as pd

df = pd.read_csv(
    "../data/raw/PPR-ALL.csv",
    encoding="cp1252",
    low_memory=False
)

print(df.shape)
df.head()

(814413, 9)


,Date of Sale (dd/mm/yyyy),Address,County,Eircode,Price (€),Not Full Market Price,VAT Exclusive,Description of Property,Property Size Description
0,01/01/2010,"5 Braemor Drive, Churchtown, Co.Dublin",Dublin,NaN,"€343,000.00",No,No,Second-Hand Dwelling house /Apartment,NaN
1,03/01/2010,"134 Ashewood Walk, Summerhill Lane, Portlaoise",Laois,NaN,"€185,000.00",No,Yes,New Dwelling house /Apartment,greater than or equal to 38 sq metres and less...
2,04/01/2010,"1 Meadow Avenue, Dundrum, Dublin 14",Dublin,NaN,"€438,500.00",No,No,Second-Hand Dwelling house /Apartment,NaN
3,04/01/2010,"1 The Haven, Mornington",Meath,NaN,"€400,000.00",No,No,Second-Hand Dwelling house /Apartment,NaN
4,04/01/2010,"11 Melville Heights, Kilkenny",Kilkenny,NaN,"€160,000.00",No,No,Second-Hand Dwelling house /Apartment,NaN


   ## Messy data notes

   **Dataset:** 814,413 sales, 9 columns (2010 – present)

   1. **Encoding:** file uses Windows `cp1252` encoding; `latin-1` broke the € symbol.
   2. **Mixed types:** "Property Size Description" has mixed text and empty values. Fixed loading with `low_memory=False`.
   3. **Price is text:** stored as "€343,000.00", needs converting to a number.
   4. **Eircode mostly missing:** empty in early rows (Eircodes only started in 2015).
   5. **Property Size Description mostly missing.**
   6. **Column names are long and messy:** e.g. "Date of Sale (dd/mm/yyyy)", "Price (€)".
   7. **All columns are text:** dates and prices need converting to proper types.
   8. **Eircode:** only 31% present (255,818 / 814,413). Can't rely on it for location.
   9. **Property Size Description:** only 6.5% present. Too sparse to use as a feature.
   10. **No bedrooms / floor area columns:** model must rely on area, date, new vs second-hand.
   11. **County is clean:** exactly 26 counties, no typos. Dublin = 31% of all sales (data is Dublin-heavy).
   12. **Not Full Market Price = Yes:** 41,490 sales (5.1%). Exclude from price analysis.
   13. **VAT Exclusive = Yes:** 143,857 sales (17.7%). Close to but not equal to new builds (146,244). Investigate.
   14. **Description of Property in two languages:** 5 different values for only 2 real categories:
      - Second-hand: "Second-Hand Dwelling house /Apartment" (668,124) + Irish "Teach/Árasán Cónaithe Atháimhe" (45) = **668,169**
      - New: "New Dwelling house /Apartment" (146,240) + Irish "Teach/Árasán Cónaithe Nua" (3) + broken-encoding "Teach/?ras?n C?naithe Nua" (1) = **146,244**
      - Map all to 2 categories: New / Second-Hand.

In [4]:
   df.info()

<class 'pandas.DataFrame'>
RangeIndex: 814413 entries, 0 to 814412
Data columns (total 9 columns):
 #   Column                     Non-Null Count   Dtype
---  ------                     --------------   -----
 0   Date of Sale (dd/mm/yyyy)  814413 non-null  str  
 1   Address                    814413 non-null  str  
 2   County                     814413 non-null  str  
 3   Eircode                    255818 non-null  str  
 4   Price (€)                  814413 non-null  str  
 5   Not Full Market Price      814413 non-null  str  
 6   VAT Exclusive              814413 non-null  str  
 7   Description of Property    814413 non-null  str  
 8   Property Size Description  52846 non-null   str  
dtypes: str(9)
memory usage: 55.9 MB


In [5]:
   for col in ["County", "Not Full Market Price", "VAT Exclusive", "Description of Property"]:
       print(f"\n--- {col} ---")
       print(df[col].value_counts())


--- County ---
County
Dublin       255242
Cork          90120
Kildare       44496
Galway        39167
Meath         33894
Limerick      29821
Wexford       28485
Wicklow       27542
Louth         22983
Waterford     22286
Kerry         22036
Tipperary     21744
Donegal       21580
Mayo          19390
Clare         18184
Westmeath     15748
Laois         13565
Kilkenny      13154
Cavan         11982
Sligo         11832
Roscommon     11461
Offaly        10429
Carlow         8862
Leitrim        7143
Longford       6856
Monaghan       6411
Name: count, dtype: int64

--- Not Full Market Price ---
Not Full Market Price
No     772923
Yes     41490
Name: count, dtype: int64

--- VAT Exclusive ---
VAT Exclusive
No     670556
Yes    143857
Name: count, dtype: int64

--- Description of Property ---
Description of Property
Second-Hand Dwelling house /Apartment    668124
New Dwelling house /Apartment            146240
Teach/Árasán Cónaithe Atháimhe               45
Teach/Árasán Cónaithe Nua       

In [6]:
   price = (
       df["Price (€)"]
       .str.replace("€", "", regex=False)
       .str.replace(",", "", regex=False)
       .astype(float)
   )

   pd.set_option("display.float_format", "{:,.0f}".format)

   print(price.describe())

count       814,413
mean        323,262
std       1,330,650
min           5,001
25%         145,161
50%         245,100
75%         370,000
max     387,665,198
Name: Price (€), dtype: float64


15. **Prices are heavily skewed:** median €245,100 vs mean €323,262. Use median for reporting, log(price) for modelling.
16. **Extreme high outliers:** max €387.7 million, almost certainly bulk / portfolio sales (e.g. whole apartment blocks), not single homes. Need to detect and exclude.
17. **Possible minimum price cutoff:** min is €5,001. Very cheap sales may be sites, partial shares or errors. Investigate.

In [7]:
   df["price_num"] = price

   top10 = df.sort_values("price_num", ascending=False).head(10)

   top10[["Date of Sale (dd/mm/yyyy)", "Address", "County", "price_num", "Description of Property"]]

,Date of Sale (dd/mm/yyyy),Address,County,price_num,Description of Property
690768,17/10/2024,"24 Tinakilly Grove, Tinakilly Park, Rathnew",Wicklow,"387,665,198",New Dwelling house /Apartment
806861,03/09/2026,"Folio DN247776F, Dublin",Dublin,"251,769,370",New Dwelling house /Apartment
586460,10/02/2023,"O'Devaney Gardens, Arbour Hill, Dublin 7",Dublin,"225,000,000",New Dwelling house /Apartment
771160,27/01/2026,"Montpelier, Dublin 7",Dublin,"225,000,000",New Dwelling house /Apartment
706229,23/12/2024,"Cooper Square, Seven Mills, Clonburris",Dublin,"221,942,731",New Dwelling house /Apartment
797943,15/07/2026,"Part Folio DN244581F comprising, 461 Apartment...",Dublin,"217,488,769",Second-Hand Dwelling house /Apartment
780906,31/03/2026,"BLOCK A B AND C, NEWMARKET YARDS",Dublin,"189,392,526",Second-Hand Dwelling house /Apartment
797944,15/07/2026,"Part of Grand Canal Harbour Apts, 1-150 Block...",Dublin,"186,544,903",Second-Hand Dwelling house /Apartment
431253,17/07/2020,"Apartments 1 - 186 Cheevers Court, Apartments ...",Dublin,"182,378,855",New Dwelling house /Apartment
809489,18/09/2026,"APT 307 BLCOK B1, 2 FISH LANE, DUBLIN LANDINGS",Dublin,"177,056,937",Second-Hand Dwelling house /Apartment


18. **Bulk / portfolio sales:** the top 10 prices (€177m – €388m) are all whole developments, not single homes. Clues: "Folio DN...", "461 Apartments", "Apts 1-150", "Block A B and C", development names with no house number.
19. **Some bulk sales look like single homes:** e.g. "24 Tinakilly Grove" (€387.7m), "APT 307 BLCOK B1" (€177m). Address keywords alone won't catch them; need price rules too.
20. **Address typos exist:** e.g. "BLCOK" instead of "BLOCK".
21. **Data is very recent:** latest sales seen in Sept 2026.

In [8]:
   for limit in [1_000_000, 2_000_000, 3_000_000, 5_000_000, 10_000_000]:
       n = (df["price_num"] > limit).sum()
       pct = n / len(df) * 100
       print(f"Over €{limit:,}: {n:,} sales ({pct:.2f}%)")

   print()
   print("99th percentile:  €{:,.0f}".format(df["price_num"].quantile(0.99)))
   print("99.9th percentile: €{:,.0f}".format(df["price_num"].quantile(0.999)))

Over €1,000,000: 16,312 sales (2.00%)
Over €2,000,000: 4,090 sales (0.50%)
Over €3,000,000: 2,054 sales (0.25%)
Over €5,000,000: 1,064 sales (0.13%)
Over €10,000,000: 511 sales (0.06%)

99th percentile:  €1,425,000
99.9th percentile: €6,333,448


22. **High-price distribution:** >€1m: 16,312 (2.0%), >€2m: 4,090 (0.5%), >€3m: 2,054 (0.25%), >€5m: 1,064 (0.13%), >€10m: 511 (0.06%). 99th percentile = €1.43m, 99.9th = €6.33m.
23. **Decision:** don't delete — add an `is_bulk_suspect` flag. Flag if address has bulk keywords (Apartments, Apts, Block, Folio, unit ranges) OR price > €5m. Exclude flagged rows from the model; test sensitivity with €3m vs €5m cutoff.

In [9]:
   for limit in [10_000, 20_000, 30_000, 50_000]:
       n = (df["price_num"] < limit).sum()
       pct = n / len(df) * 100
       print(f"Under €{limit:,}: {n:,} sales ({pct:.2f}%)")

   print()
   cheap = df[df["price_num"] < 20_000]
   print("Of sales under €20k, marked 'Not Full Market Price':")
   print(cheap["Not Full Market Price"].value_counts())

   cheap[["Date of Sale (dd/mm/yyyy)", "Address", "County", "price_num", "Not Full Market Price"]].head(10)

Under €10,000: 1,103 sales (0.14%)
Under €20,000: 4,924 sales (0.60%)
Under €30,000: 11,957 sales (1.47%)
Under €50,000: 34,858 sales (4.28%)

Of sales under €20k, marked 'Not Full Market Price':
Not Full Market Price
No     3877
Yes    1047
Name: count, dtype: int64


,Date of Sale (dd/mm/yyyy),Address,County,price_num,Not Full Market Price
203,08/01/2010,"Number 8, Florence Road, Bray",Wicklow,"16,000",No
204,08/01/2010,"Rathquarter, Sligo",Sligo,"12,000",Yes
344,14/01/2010,"31 dromard road, drimnagh",Dublin,"16,000",Yes
378,14/01/2010,"Srabra, Lough Allen, Boyle",Roscommon,"5,500",Yes
493,18/01/2010,"30 Chapel Grove, Balbriggan, Co Dublin",Dublin,"10,000",Yes
605,20/01/2010,"Ardaneer, Foynes, Co. Limerick",Limerick,"12,500",No
607,20/01/2010,"Caherline, Caherconlish",Limerick,"18,417",No
1002,28/01/2010,"90 Maudlin Court, Thomastown",Kilkenny,"10,000",No
1005,28/01/2010,"97 Great William O'Brien Street, Blackpool, Cork",Cork,"6,000",No
1205,02/02/2010,"20 Royal Canal Court, Ashtown, Dublin 15",Dublin,"17,667",No


24. **Cheap-end distribution:** <€10k: 1,103 (0.14%), <€20k: 4,924 (0.60%), <€30k: 11,957 (1.47%), <€50k: 34,858 (4.28%).
25. **"Not Full Market Price" flag is unreliable:** only 21% of sales under €20k are flagged (1,047 of 4,924). Many unflagged ones look like normal homes at impossible prices (e.g. Florence Road, Bray: €16,000).
26. **Likely partial / share sales:** odd non-round prices (€17,667, €18,417) suggest fractions of a full price (e.g. €53,000 ÷ 3). Some cheap rural sales may be genuine (derelict cottages, sites), especially 2010–2013.
27. **Decision:** add an `is_low_price_suspect` flag for sales < €20,000. Exclude from the model along with bulk suspects and Not Full Market Price = Yes.